# UC13 — Clasificación Automática de Llamadas

Pipeline de clasificación multi-clase de motivos de llamada a partir de transcripciones.

**Valor de negocio**: Routing inteligente, reducción de AHT y reporting preciso sin tags manuales.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS CLASIFICACION_LLAMADAS_DB;
USE SCHEMA CLASIFICACION_LLAMADAS_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Transcripciones Sintéticas (1.000 llamadas)

6 categorías: Siniestro, Cambio_Poliza, Cancelacion, Facturacion, Consulta_Cobertura, Asistencia.

In [ ]:
CREATE OR REPLACE TABLE llamadas AS
SELECT 'CALL' || LPAD(SEQ4(),5,'0') AS llamada_id,
    DATEADD(day,-UNIFORM(1,180,RANDOM()),CURRENT_DATE()) AS fecha,
    'AGT' || LPAD(MOD(SEQ4(),20)+1,3,'0') AS agente_id,
    UNIFORM(60,900,RANDOM()) AS duracion_seg,
    CASE MOD(SEQ4(),6)
        WHEN 0 THEN 'Siniestro'
        WHEN 1 THEN 'Cambio_Poliza'
        WHEN 2 THEN 'Cancelacion'
        WHEN 3 THEN 'Facturacion'
        WHEN 4 THEN 'Consulta_Cobertura'
        ELSE 'Asistencia'
    END AS motivo_real,
    CASE MOD(SEQ4(),6)
        WHEN 0 THEN CASE MOD(SEQ4(),4) WHEN 0 THEN 'Buenos días, le llamo porque tuve un accidente de tráfico ayer por la tarde. Otro coche me dio por detrás en un semáforo. Necesito abrir un parte de siniestro y que me envíen una grúa.' WHEN 1 THEN 'Hola, me han robado el coche esta noche del garaje. He puesto la denuncia en la policía y necesito comunicar el siniestro urgentemente.' WHEN 2 THEN 'Llamo porque se me ha inundado la cocina por una rotura de tubería. Hay daños en el suelo y en los muebles. ¿Cómo debo proceder con el seguro del hogar?' ELSE 'He tenido un accidente en la autopista. El otro conductor se dio a la fuga. Tengo fotos y testigos. Quiero declarar el siniestro.' END
        WHEN 1 THEN CASE MOD(SEQ4(),3) WHEN 0 THEN 'Quiero cambiar el vehículo de mi póliza de auto. He vendido el anterior y he comprado uno nuevo. ¿Qué documentación necesitan?' WHEN 1 THEN 'Necesito añadir un segundo conductor a mi póliza. Es mi hijo que acaba de sacarse el carnet.' ELSE 'Me he mudado de piso y necesito actualizar la dirección de mi seguro de hogar y revisar las coberturas.' END
        WHEN 2 THEN CASE MOD(SEQ4(),3) WHEN 0 THEN 'Quiero cancelar mi póliza de auto. He encontrado un precio mejor en otra compañía y no estoy satisfecho con el servicio.' WHEN 1 THEN 'Llamo para dar de baja el seguro de hogar. Ya no vivo en ese piso y no necesito la cobertura.' ELSE 'Estoy pensando en cancelar. Llevo esperando 2 meses la resolución de mi siniestro y nadie me informa. Si no hay solución, me voy.' END
        WHEN 3 THEN CASE MOD(SEQ4(),3) WHEN 0 THEN 'No entiendo el último recibo que me han cobrado. La prima ha subido un 20% respecto al año pasado sin previo aviso.' WHEN 1 THEN 'Me han cobrado dos veces el recibo de este mes. Necesito que me devuelvan el importe duplicado urgentemente.' ELSE 'Quiero cambiar la forma de pago de anual a mensual. ¿Tiene algún coste adicional?' END
        WHEN 4 THEN CASE MOD(SEQ4(),3) WHEN 0 THEN '¿Mi seguro de auto cubre los daños por granizo? Ayer cayó una granizada fuerte y mi coche tiene abolladuras.' WHEN 1 THEN 'Quiero saber si mi póliza de hogar cubre los daños causados por mi perro al vecino de abajo.' ELSE 'Necesito información sobre las coberturas de mi seguro de vida. ¿Incluye invalidez permanente?' END
        ELSE CASE MOD(SEQ4(),3) WHEN 0 THEN 'Necesito asistencia en carretera. Mi coche se ha averiado en la A-6 a la altura del kilómetro 45. No arranca.' WHEN 1 THEN 'He tenido un pinchazo en la autopista y no tengo rueda de repuesto. Necesito una grúa urgente.' ELSE 'Mi coche no arranca y tengo que ir al hospital a una cita urgente. ¿Me pueden enviar un taxi?' END
    END AS transcripcion
FROM TABLE(GENERATOR(ROWCOUNT=>1000));

SELECT motivo_real, COUNT(*) AS llamadas, ROUND(AVG(duracion_seg)) AS duracion_media FROM llamadas GROUP BY motivo_real ORDER BY llamadas DESC;

---

## Paso 3: Clasificar Motivos con Cortex AI

In [ ]:
CREATE OR REPLACE TABLE llamadas_clasificadas AS
SELECT l.*,
    TRIM(SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Clasifica esta transcripción de llamada de seguros en exactamente UNA categoría. Responde SOLO con la categoría sin explicaciones: Siniestro, Cambio_Poliza, Cancelacion, Facturacion, Consulta_Cobertura, Asistencia.\n\nTranscripción: ' || transcripcion
    )) AS motivo_predicho
FROM llamadas l;

SELECT motivo_real, motivo_predicho, COUNT(*) AS total
FROM llamadas_clasificadas GROUP BY motivo_real, motivo_predicho ORDER BY motivo_real, total DESC;

---

## Paso 4: Extraer Subclasificación y Urgencia

In [ ]:
CREATE OR REPLACE TABLE llamadas_enriquecidas AS
SELECT lc.*,
    TRIM(SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Analiza esta transcripción de llamada de seguros y devuelve SOLO un JSON: {"urgencia": "alta/media/baja", "sentimiento": "positivo/neutro/negativo", "requiere_escalado": true/false, "resumen_corto": "máximo 15 palabras"}.\n\nTranscripción: ' || transcripcion
    )) AS analisis_raw,
    SNOWFLAKE.CORTEX.SENTIMENT(transcripcion) AS sentimiento_score
FROM llamadas_clasificadas lc;

SELECT llamada_id, motivo_predicho, sentimiento_score, LEFT(analisis_raw,200) AS analisis
FROM llamadas_enriquecidas LIMIT 10;

---

## Paso 5: Análisis de Routing y Tiempos

In [ ]:
SELECT motivo_predicho AS motivo, COUNT(*) AS llamadas,
    ROUND(AVG(duracion_seg)) AS duracion_media_seg,
    ROUND(AVG(sentimiento_score),3) AS sentimiento_medio,
    SUM(CASE WHEN sentimiento_score < -0.3 THEN 1 ELSE 0 END) AS llamadas_negativas
FROM llamadas_enriquecidas GROUP BY motivo_predicho ORDER BY llamadas DESC;

---

## Paso 6: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Clasificación Automática de Llamadas')
st.markdown('### Routing Inteligente — Snowflake Cortex AI')

df = session.table('llamadas_enriquecidas').to_pandas()

c1,c2,c3,c4 = st.columns(4)
with c1: st.metric('Total Llamadas', len(df))
with c2: st.metric('Categorías', df['MOTIVO_PREDICHO'].nunique())
with c3: st.metric('Duración Media', f"{df['DURACION_SEG'].mean():.0f}s")
with c4: st.metric('Sentimiento Medio', f"{df['SENTIMIENTO_SCORE'].mean():.2f}")

st.markdown('---')
st.subheader('Llamadas por Motivo')
rc = df['MOTIVO_PREDICHO'].value_counts()
st.bar_chart(pd.DataFrame({'Llamadas': rc.values}, index=rc.index))

st.markdown('---')
st.subheader('Precisión de Clasificación')
acc = (df['MOTIVO_REAL']==df['MOTIVO_PREDICHO']).mean()
st.metric('Exactitud Global', f'{acc:.1%}')

st.markdown('---')
motivo = st.selectbox('Filtrar por motivo', ['Todos']+list(df['MOTIVO_PREDICHO'].unique()))
fdf = df if motivo=='Todos' else df[df['MOTIVO_PREDICHO']==motivo]
st.dataframe(fdf[['LLAMADA_ID','FECHA','MOTIVO_PREDICHO','MOTIVO_REAL','DURACION_SEG','SENTIMIENTO_SCORE']].head(100), use_container_width=True, height=400)
st.caption('Desarrollado con Snowflake Cortex AI + Streamlit')

---

## Paso 7: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS CLASIFICACION_LLAMADAS_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;